# Monte Carlo Simulation: OLS vs Newey–West HAC under AR(4) Autocorrelation

Goal: Evaluate the performance of ordinary least squares (OLS) standard errors 
compared to heteroskedasticity-and-autocorrelation consistent (HAC) 
Newey–West estimators when residuals follow an AR(4) process.


In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
from scipy.stats import norm, chi2, t


In [25]:
# Simulation parameters
np.random.seed(2025)
R = 1000                # number of replications
T = 200                 # sample size
betas_true = np.array([0.0, 1.0, 0.5, -0.5])
phi = np.array([0.4, -0.2, 0.15, -0.05])  # AR(4) coefficients
bandwidths = [0, 1, 4, int(4 * (T/100)**(2/9))]  # HAC lag lengths


In [26]:
def simulate_ar4(T, phi, sigma=1.0):
    epsilon = np.random.normal(0, sigma, T)
    u = np.zeros(T)
    for t in range(4, T):
        u[t] = phi[0]*u[t-1] + phi[1]*u[t-2] + phi[2]*u[t-3] + phi[3]*u[t-4] + epsilon[t]
    return u

def one_replication(T, phi, betas, bandwidths):
    """
    Run one Monte Carlo replication comparing OLS and Newey-West estimators.
    
    Returns a dictionary with: 
    betahat, sandwich variance, beta variance, confidence intervals, and p-values.
    """

    # --- 1️⃣ Simulate regressors ---
    X = np.column_stack([
        np.ones(T),
        np.random.normal(size=T),
        np.random.normal(size=T),
        np.random.normal(size=T)
    ])

    # --- 2️⃣ Simulate AR(4) errors ---
    u = simulate_ar4(T,phi)

    # --- 3️⃣ Generate dependent variable ---
    y = np.matmul(X,betas) + u

    # --- 4️⃣ Fit OLS model ---
    model = sm.OLS(y, X).fit()
    betahat = model.params
    k = len(betahat)
    df = T - k
    t_crit = t.ppf(0.975, df)

    # --- 5️⃣ Initialize results dictionary ---
    results = {}

    # --- 6️⃣ Compute OLS quantities ---
    cov_ols = model.cov_params()
    se_ols = np.sqrt(np.diag(cov_ols))
    t_ols = betahat / se_ols
    p_ols = 2 * (1 - t.cdf(np.abs(t_ols), df))
    ci_ols = np.column_stack([betahat - t_crit*se_ols, betahat + t_crit*se_ols])

    results['OLS'] = {
        'betahat': betahat,
        'var_sandwich': cov_ols,
        'var_beta': np.diag(cov_ols),
        'conf_int': ci_ols,
        'pvalues': p_ols
    }

    # --- 7️⃣ Compute Newey–West quantities ---
    for m in bandwidths:
        cov_nw = cov_hac(model, nlags=m)
        se_nw = np.sqrt(np.diag(cov_nw))
        t_nw = betahat / se_nw
        p_nw = 2 * (1 - t.cdf(np.abs(t_nw), df))
        ci_nw = np.column_stack([betahat - t_crit*se_nw, betahat + t_crit*se_nw])

        results[f'NW({m})'] = {
            'betahat': betahat,
            'var_sandwich': cov_nw,
            'var_beta': np.diag(cov_nw),
            'conf_int': ci_nw,
            'pvalues': p_nw
        }

    return results



In [34]:
results = one_replication(T, phi, betas_true, bandwidths)
results["OLS"]

{'betahat': array([ 0.1307705 ,  1.07551734,  0.33252295, -0.58321584]),
 'var_sandwich': array([[ 0.00588427,  0.00041768,  0.00040926, -0.00066878],
        [ 0.00041768,  0.00570861, -0.00015307,  0.00042602],
        [ 0.00040926, -0.00015307,  0.00495508, -0.0010134 ],
        [-0.00066878,  0.00042602, -0.0010134 ,  0.00584304]]),
 'var_beta': array([0.00588427, 0.00570861, 0.00495508, 0.00584304]),
 'conf_int': array([[-0.0205105 ,  0.28205149],
        [ 0.92651155,  1.22452313],
        [ 0.19369934,  0.47134656],
        [-0.73396586, -0.43246581]]),
 'pvalues': array([8.98235677e-02, 0.00000000e+00, 4.40499821e-06, 9.99866856e-13])}

In [28]:
results

{'OLS': {'betahat': array([ 0.0324019 ,  1.13205216,  0.63342653, -0.49638106]),
  'var_sandwich': array([[ 0.0053254 ,  0.00026501,  0.00075846,  0.000121  ],
         [ 0.00026501,  0.00579058,  0.00047212, -0.00034692],
         [ 0.00075846,  0.00047212,  0.006088  , -0.00032553],
         [ 0.000121  , -0.00034692, -0.00032553,  0.00564129]]),
  'var_beta': array([0.0053254 , 0.00579058, 0.006088  , 0.00564129]),
  'conf_int': array([[-0.11151583,  0.17631963],
         [ 0.98198036,  1.28212396],
         [ 0.479549  ,  0.78730406],
         [-0.64450567, -0.34825645]]),
  'pvalues': array([6.57524118e-01, 0.00000000e+00, 5.15143483e-14, 3.57541996e-10])},
 'NW(0)': {'betahat': array([ 0.0324019 ,  1.13205216,  0.63342653, -0.49638106]),
  'var_sandwich': array([[ 0.00505693,  0.0003783 , -0.00035935,  0.000389  ],
         [ 0.0003783 ,  0.00707067,  0.00212454,  0.00022966],
         [-0.00035935,  0.00212454,  0.00580528,  0.0001664 ],
         [ 0.000389  ,  0.00022966,  0.00